# Primeiro Modelo CNN (Multissatelite)

Este notebook foi organizado para executar **um satelite por vez**, nesta ordem:
1. Sentinel-2 (itens 1 a 5)
2. Landsat 8/9 (itens 1 a 5)
3. MODIS (itens 1 a 5)

Assim evitamos estourar RAM ao carregar todos os datasets simultaneamente.


## Reuso de codigo do projeto

Funcoes reaproveitadas de `src/`:
- `src.models.cnn_builder.build_cnn_model`
- `src.models.cnn_builder.get_model_architecture_summary`
- `src.models.cnn_tf_data_pipeline.build_train_val_tf_data`
- `src.models.cnn_tf_data_pipeline.build_tf_data_pipeline`
- `src.models.cnn_data_prep.labels_from_extracted_codes`
- `src.train.callbacks.get_training_callbacks`
- `src.utils.metrics.classification_metrics_extended`


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import json
import gc
import math
import random
import time
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

try:
    import pyarrow.parquet as pq
except Exception:
    pq = None

try:
    import tensorflow as tf
except Exception as exc:
    raise ImportError(
        'TensorFlow nao esta instalado. Instale com: pip install tensorflow'
    ) from exc

repo_root = Path.cwd().resolve()
if not (repo_root / 'src').exists() and (repo_root.parent / 'src').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.models.cnn_builder import build_cnn_model, get_model_architecture_summary
from src.models.cnn_tf_data_pipeline import build_train_val_tf_data, build_tf_data_pipeline
from src.models.cnn_data_prep import labels_from_extracted_codes
from src.train.callbacks import get_training_callbacks
from src.utils.metrics import classification_metrics_extended

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 6)

print('Repo root:', repo_root)
print('TensorFlow:', tf.__version__)


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

OUTPUT_ROOT = repo_root / 'outputs' / 'a05_cnn_multissatelite'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

CODES_PATH = repo_root / 'data' / 'extracted_codes.json'
MLP_BASELINE_PATH = repo_root / 'outputs' / 'a03_mlp_baseline' / 'mlp_baseline_metrics.json'

SENSORS = {
    'sentinel2': {
        'display_name': 'Sentinel-2',
        'parquet_path': repo_root / 'data' / 'parquet' / 'sentinel_completo.parquet',
    },
    'landsat89': {
        'display_name': 'Landsat 8/9',
        'parquet_path': repo_root / 'data' / 'parquet' / 'landsat89_completo.parquet',
    },
    'modis': {
        'display_name': 'MODIS',
        'parquet_path': repo_root / 'data' / 'parquet' / 'modis_completo.parquet',
    },
}

DATA_CFG = {
    'image_col': 'image_id',
    'pixel_col': 'pixel_idx',
    'band_prefix': 'B',
    'max_images_per_sensor': None,
    'min_images_per_class': 20,
    'test_size': 0.15,
    'val_size': 0.15,
}

TRAIN_CFG = {
    'epochs': 40,
    'batch_size': 32,
    'learning_rate': 1e-3,
    'normalization': 'zscore',
    'augment_train': True,
    'patience': 8,
    'min_delta': 1e-4,
    'reduce_lr_patience': 4,
    'reduce_lr_factor': 0.5,
    'min_lr': 1e-6,
    'conv1_filters': 32,
    'conv2_filters': 64,
    'kernel_size': (3, 3),
    'dense_units': 128,
    'dropout_rate': 0.5,
    'conv_dropout_rate': 0.2,
    'l2_regularizer': 1e-3,
}

RUN_TRAINING = True

assert CODES_PATH.exists(), f'Arquivo nao encontrado: {CODES_PATH}'
print('Output root:', OUTPUT_ROOT)


## Funcoes auxiliares (dados, treino e avaliacao)

A preparacao abaixo converte o formato longo para tensor 4D por `image_id`.
A funcao `run_sensor_pipeline` executa as 5 etapas para um unico satelite
(e limpa memoria ao final).


In [ ]:
def to_python(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {k: to_python(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_python(v) for v in obj]
    if isinstance(obj, tuple):
        return [to_python(v) for v in obj]
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


def save_json(path: Path, payload: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        json.dump(to_python(payload), f, ensure_ascii=False, indent=2)


def band_sort_key(name: str):
    import re

    m = re.match(r'^([A-Za-z]+)(\d+)([A-Za-z]*)$', name)
    if not m:
        return (name, 0, '')
    prefix, num, suffix = m.groups()
    return (prefix, int(num), suffix)


def infer_band_columns(columns: list[str], band_prefix: str = 'B') -> list[str]:
    bands = [c for c in columns if c.startswith(band_prefix)]
    bands = [b for b in bands if any(ch.isdigit() for ch in b)]
    return sorted(bands, key=band_sort_key)


def get_parquet_columns(parquet_path: Path) -> list[str]:
    if pq is not None:
        return list(pq.ParquetFile(parquet_path).schema.names)
    return list(pd.read_parquet(parquet_path).columns)


def infer_hw(n_pixels: int) -> tuple[int, int]:
    side = int(math.sqrt(n_pixels))
    if side * side == n_pixels:
        return side, side

    best_h, best_w = 1, n_pixels
    best_gap = abs(best_w - best_h)
    for h in range(1, int(math.sqrt(n_pixels)) + 1):
        if n_pixels % h == 0:
            w = n_pixels // h
            gap = abs(w - h)
            if gap < best_gap:
                best_h, best_w, best_gap = h, w, gap
    return best_h, best_w


def summarize_image_pixels(meta_df: pd.DataFrame, image_col: str, pixel_col: str | None) -> pd.Series:
    if pixel_col and pixel_col in meta_df.columns:
        return meta_df.groupby(image_col)[pixel_col].nunique().rename('n_pixels')
    return meta_df.groupby(image_col).size().rename('n_pixels')


def read_rows_for_image_ids(
    parquet_path: Path,
    selected_ids: list[str],
    columns: list[str],
    image_col: str,
    batch_size: int = 200_000,
) -> pd.DataFrame:
    selected_ids = set(map(str, selected_ids))

    if pq is None:
        df = pd.read_parquet(parquet_path, columns=columns)
        return df[df[image_col].astype(str).isin(selected_ids)].copy()

    pf = pq.ParquetFile(parquet_path)
    chunks = []
    for batch in pf.iter_batches(columns=columns, batch_size=batch_size):
        chunk = batch.to_pandas()
        chunk = chunk[chunk[image_col].astype(str).isin(selected_ids)]
        if not chunk.empty:
            chunks.append(chunk)

    if not chunks:
        return pd.DataFrame(columns=columns)
    return pd.concat(chunks, ignore_index=True)


def build_image_tensor_from_long(
    df_rows: pd.DataFrame,
    image_ids: list[str],
    band_cols: list[str],
    target_pixels: int,
    height: int,
    width: int,
    image_col: str,
    pixel_col: str | None,
) -> np.ndarray:
    n_images = len(image_ids)
    n_bands = len(band_cols)

    x = np.zeros((n_images, height, width, n_bands), dtype=np.float32)
    grouped = {str(k): v for k, v in df_rows.groupby(image_col, sort=False)}

    for i, img_id in enumerate(image_ids):
        group = grouped.get(str(img_id))
        if group is None:
            continue

        flat = np.zeros((target_pixels, n_bands), dtype=np.float32)
        values = group[band_cols].to_numpy(dtype=np.float32, copy=False)

        if pixel_col and pixel_col in group.columns:
            idx = group[pixel_col].to_numpy(dtype=np.int64, copy=False)
            valid = (idx >= 0) & (idx < target_pixels)
            flat[idx[valid]] = values[valid]
        else:
            n_fill = min(target_pixels, len(values))
            flat[:n_fill] = values[:n_fill]

        x[i] = flat.reshape(height, width, n_bands)

    return x


def prepare_long_sensor_dataset(
    parquet_path: Path,
    codes_path: Path,
    sensor_name: str,
    image_col: str,
    pixel_col: str,
    band_prefix: str,
    max_images: int | None,
    min_images_per_class: int,
    seed: int,
) -> dict[str, Any]:
    if not parquet_path.exists():
        raise FileNotFoundError(f'Parquet nao encontrado para {sensor_name}: {parquet_path}')

    all_columns = get_parquet_columns(parquet_path)
    band_cols = infer_band_columns(all_columns, band_prefix=band_prefix)
    if not band_cols:
        raise ValueError(f'{sensor_name}: nenhuma coluna de banda encontrada.')

    meta_cols = [image_col]
    has_pixel_col = pixel_col in all_columns
    if has_pixel_col:
        meta_cols.append(pixel_col)

    meta_df = pd.read_parquet(parquet_path, columns=meta_cols)
    meta_df[image_col] = meta_df[image_col].astype(str)

    pixel_counts = summarize_image_pixels(meta_df, image_col=image_col, pixel_col=pixel_col if has_pixel_col else None)
    target_pixels = int(pixel_counts.mode().iloc[0])

    valid_ids = pixel_counts[pixel_counts == target_pixels].index.astype(str).tolist()
    if max_images is not None and len(valid_ids) > max_images:
        rng = np.random.default_rng(seed)
        valid_ids = rng.choice(valid_ids, size=max_images, replace=False).tolist()

    selected_columns = [image_col] + ([pixel_col] if has_pixel_col else []) + band_cols
    rows_df = read_rows_for_image_ids(
        parquet_path=parquet_path,
        selected_ids=valid_ids,
        columns=selected_columns,
        image_col=image_col,
    )
    rows_df[image_col] = rows_df[image_col].astype(str)

    height, width = infer_hw(target_pixels)
    x_all = build_image_tensor_from_long(
        df_rows=rows_df,
        image_ids=valid_ids,
        band_cols=band_cols,
        target_pixels=target_pixels,
        height=height,
        width=width,
        image_col=image_col,
        pixel_col=pixel_col if has_pixel_col else None,
    )

    y_all, _ = labels_from_extracted_codes(valid_ids, codes_path)
    label_mask = y_all != -1

    x = x_all[label_mask]
    y = y_all[label_mask].astype(np.int64)
    image_ids = np.array(valid_ids, dtype=object)[label_mask]

    class_counts = pd.Series(y).value_counts().to_dict()
    if 0 not in class_counts or 1 not in class_counts:
        raise ValueError(f'{sensor_name}: faltou classe positiva/negativa apos filtragem.')
    if min(class_counts.values()) < min_images_per_class:
        raise ValueError(f'{sensor_name}: poucas imagens por classe ({class_counts}).')

    summary = {
        'sensor_name': sensor_name,
        'parquet_path': str(parquet_path),
        'n_bands': int(len(band_cols)),
        'band_cols': band_cols,
        'height': int(height),
        'width': int(width),
        'target_pixels': int(target_pixels),
        'n_images_after_label_filter': int(len(image_ids)),
        'class_balance': {str(int(k)): int(v) for k, v in pd.Series(y).value_counts().sort_index().items()},
        'tensor_shape': list(x.shape),
    }

    return {
        'X': x,
        'y': y,
        'image_ids': image_ids,
        'summary': summary,
    }


def one_hot(y: np.ndarray, n_classes: int = 2) -> np.ndarray:
    return tf.keras.utils.to_categorical(y.astype(np.int64), num_classes=n_classes)


def split_train_val_test(
    x: np.ndarray,
    y: np.ndarray,
    image_ids: np.ndarray,
    test_size: float,
    val_size: float,
    seed: int,
) -> dict[str, tuple[np.ndarray, np.ndarray, np.ndarray]]:
    idx = np.arange(len(y))
    temp_size = test_size + val_size
    if not (0 < temp_size < 1):
        raise ValueError('test_size + val_size deve estar entre 0 e 1.')

    train_idx, temp_idx = train_test_split(
        idx,
        test_size=temp_size,
        stratify=y,
        random_state=seed,
    )

    temp_y = y[temp_idx]
    relative_test = test_size / temp_size

    val_idx, test_idx = train_test_split(
        temp_idx,
        test_size=relative_test,
        stratify=temp_y,
        random_state=seed,
    )

    return {
        'train': (x[train_idx], y[train_idx], image_ids[train_idx]),
        'val': (x[val_idx], y[val_idx], image_ids[val_idx]),
        'test': (x[test_idx], y[test_idx], image_ids[test_idx]),
    }


def plot_history(history: dict[str, list[float]], title_prefix: str):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.get('loss', []), label='train_loss')
    axes[0].plot(history.get('val_loss', []), label='val_loss')
    axes[0].set_title(f'{title_prefix} - Loss')
    axes[0].set_xlabel('epoca')
    axes[0].legend()

    axes[1].plot(history.get('accuracy', []), label='train_acc')
    axes[1].plot(history.get('val_accuracy', []), label='val_acc')
    axes[1].set_title(f'{title_prefix} - Accuracy')
    axes[1].set_xlabel('epoca')
    axes[1].legend()

    plt.tight_layout()
    plt.show()


def plot_conf_matrix(y_true: np.ndarray, y_pred: np.ndarray, title: str):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['negativo', 'positivo'])
    disp.plot(cmap='Blues', values_format='d')
    plt.title(title)
    plt.grid(False)
    plt.show()


def load_mlp_baseline_metrics(path: Path) -> dict[str, float | None]:
    if not path.exists():
        return {'accuracy': None, 'loss': None}

    with path.open('r', encoding='utf-8') as f:
        raw = json.load(f)

    acc = raw.get('accuracy')
    if acc is None:
        acc = raw.get('compile_metrics')

    return {
        'accuracy': float(acc) if acc is not None else None,
        'loss': float(raw['loss']) if 'loss' in raw else None,
    }


def run_sensor_pipeline(
    sensor_key: str,
    sensor_cfg: dict[str, Any],
    data_cfg: dict[str, Any],
    train_cfg: dict[str, Any],
    codes_path: Path,
    output_root: Path,
    seed: int = 42,
    show_plots: bool = True,
) -> dict[str, Any] | None:
    sensor_name = sensor_cfg['display_name']
    parquet_path = sensor_cfg['parquet_path']

    if not parquet_path.exists():
        print(f'[SKIP] {sensor_name}: parquet ausente em {parquet_path}')
        return None

    print(f'\n===== {sensor_name} =====')
    print('(1) Preparacao dos dados...')

    prep = prepare_long_sensor_dataset(
        parquet_path=parquet_path,
        codes_path=codes_path,
        sensor_name=sensor_name,
        image_col=data_cfg['image_col'],
        pixel_col=data_cfg['pixel_col'],
        band_prefix=data_cfg['band_prefix'],
        max_images=data_cfg['max_images_per_sensor'],
        min_images_per_class=data_cfg['min_images_per_class'],
        seed=seed,
    )

    x = prep['X']
    y = prep['y']
    image_ids = prep['image_ids']

    splits = split_train_val_test(
        x=x,
        y=y,
        image_ids=image_ids,
        test_size=data_cfg['test_size'],
        val_size=data_cfg['val_size'],
        seed=seed,
    )

    x_train, y_train, ids_train = splits['train']
    x_val, y_val, ids_val = splits['val']
    x_test, y_test, ids_test = splits['test']

    y_train_oh = one_hot(y_train, n_classes=2)
    y_val_oh = one_hot(y_val, n_classes=2)
    y_test_oh = one_hot(y_test, n_classes=2)

    print('(2) Arquitetura CNN + (3) Treino e validacao...')
    data_bundle = build_train_val_tf_data(
        x_train=x_train,
        y_train=y_train_oh,
        x_val=x_val,
        y_val=y_val_oh,
        batch_size=train_cfg['batch_size'],
        normalization=train_cfg['normalization'],
        data_format='channels_last',
        target_channels=x_train.shape[-1],
        augment_train=train_cfg['augment_train'],
        seed=seed,
    )

    test_ds, _ = build_tf_data_pipeline(
        x=x_test,
        y=y_test_oh,
        batch_size=train_cfg['batch_size'],
        training=False,
        shuffle=False,
        normalization=train_cfg['normalization'],
        normalizer=data_bundle['normalizer'],
        data_format='channels_last',
        target_channels=x_test.shape[-1],
    )

    model = build_cnn_model(
        input_shape=tuple(data_bundle['train_meta']['input_shape']),
        n_classes=2,
        conv1_filters=train_cfg['conv1_filters'],
        conv2_filters=train_cfg['conv2_filters'],
        kernel_size=tuple(train_cfg['kernel_size']),
        dense_units=train_cfg['dense_units'],
        dropout_rate=train_cfg['dropout_rate'],
        conv_dropout_rate=train_cfg['conv_dropout_rate'],
        l2_regularizer=train_cfg['l2_regularizer'],
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=train_cfg['learning_rate']),
        loss=model.loss,
        metrics=['accuracy'],
    )

    sensor_dir = output_root / sensor_key
    sensor_dir.mkdir(parents=True, exist_ok=True)

    callbacks = get_training_callbacks(
        checkpoint_path=str(sensor_dir / 'best_model.keras'),
        monitor='val_loss',
        patience=train_cfg['patience'],
        min_delta=train_cfg['min_delta'],
        extra_callbacks=[
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=train_cfg['reduce_lr_factor'],
                patience=train_cfg['reduce_lr_patience'],
                min_lr=train_cfg['min_lr'],
                verbose=1,
            )
        ],
    )

    t0 = time.perf_counter()
    history = model.fit(
        data_bundle['train_ds'],
        validation_data=data_bundle['val_ds'],
        epochs=train_cfg['epochs'],
        callbacks=callbacks,
        verbose=1,
    )
    train_time_s = time.perf_counter() - t0

    ckpt_path = sensor_dir / 'best_model.keras'
    best_model = tf.keras.models.load_model(ckpt_path) if ckpt_path.exists() else model

    print('(4) Avaliacao quantitativa...')
    eval_dict = best_model.evaluate(test_ds, return_dict=True, verbose=0)
    y_prob = best_model.predict(test_ds, verbose=0)
    y_score = y_prob[:, 1] if y_prob.ndim == 2 else y_prob.ravel()
    y_pred = np.argmax(y_prob, axis=1) if y_prob.ndim == 2 else (y_score >= 0.5).astype(int)

    clf = classification_metrics_extended(y_test, y_pred, y_score)

    final_train_acc = float(history.history['accuracy'][-1]) if history.history.get('accuracy') else None
    final_val_acc = float(history.history['val_accuracy'][-1]) if history.history.get('val_accuracy') else None
    gap = None if (final_train_acc is None or final_val_acc is None) else (final_train_acc - final_val_acc)

    split_summary = {
        'n_train': int(len(y_train)),
        'n_val': int(len(y_val)),
        'n_test': int(len(y_test)),
        'class_balance_train': {str(int(k)): int(v) for k, v in pd.Series(y_train).value_counts().sort_index().items()},
        'class_balance_val': {str(int(k)): int(v) for k, v in pd.Series(y_val).value_counts().sort_index().items()},
        'class_balance_test': {str(int(k)): int(v) for k, v in pd.Series(y_test).value_counts().sort_index().items()},
    }

    metrics_out = {
        'sensor_key': sensor_key,
        'sensor_name': sensor_name,
        'train_time_s': float(train_time_s),
        'keras_test_metrics': eval_dict,
        'classification_metrics_test': clf,
        'generalization_gap_train_minus_val_acc': gap,
        'epochs_ran': int(len(history.history.get('loss', []))),
        'input_shape': list(data_bundle['train_meta']['input_shape']),
        'split_summary': split_summary,
    }

    architecture = get_model_architecture_summary(model)

    save_json(sensor_dir / 'data_prep_summary.json', prep['summary'])
    save_json(sensor_dir / 'split_summary.json', split_summary)
    save_json(sensor_dir / 'architecture_summary.json', architecture)
    save_json(sensor_dir / 'history.json', history.history)
    save_json(sensor_dir / 'metrics_test.json', metrics_out)

    pd.DataFrame(history.history).to_csv(sensor_dir / 'history.csv', index=False)
    pd.DataFrame(confusion_matrix(y_test, y_pred)).to_csv(sensor_dir / 'confusion_matrix.csv', index=False)

    if show_plots:
        plot_history(history.history, title_prefix=sensor_name)
        plot_conf_matrix(y_test, y_pred, title=f'{sensor_name} - Matriz de confusao (teste)')

    print('(5) Analise critica local:')
    if gap is not None and gap > 0.08:
        print(f'- Possivel overfitting (gap treino-validacao={gap:.4f}).')
    else:
        print(f'- Gap treino-validacao controlado ({gap:.4f}).' if gap is not None else '- Gap indisponivel.')
    print(f"- F1 teste: {clf.get('f1', float('nan')):.4f}")
    print(f"- Accuracy teste: {clf.get('accuracy', float('nan')):.4f}")

    result = {
        'sensor_key': sensor_key,
        'sensor_name': sensor_name,
        'accuracy': clf.get('accuracy'),
        'f1': clf.get('f1'),
        'roc_auc': clf.get('roc_auc'),
        'pr_auc': clf.get('pr_auc'),
        'gap': gap,
        'train_time_s': train_time_s,
    }

    # Limpeza explicita de memoria (importante para nao somar datasets em RAM)
    del x, y, image_ids
    del x_train, y_train, ids_train
    del x_val, y_val, ids_val
    del x_test, y_test, ids_test
    del y_train_oh, y_val_oh, y_test_oh
    del data_bundle, test_ds, model, best_model, history, y_prob, y_score, y_pred

    tf.keras.backend.clear_session()
    gc.collect()

    return result


def collect_saved_metrics(output_root: Path, sensors: dict[str, dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for sensor_key, sensor_cfg in sensors.items():
        p = output_root / sensor_key / 'metrics_test.json'
        if not p.exists():
            continue
        m = json.loads(p.read_text(encoding='utf-8'))
        clf = m.get('classification_metrics_test', {})
        keras_m = m.get('keras_test_metrics', {})
        rows.append({
            'sensor_key': sensor_key,
            'sensor': sensor_cfg['display_name'],
            'cnn_accuracy': clf.get('accuracy'),
            'cnn_precision': clf.get('precision'),
            'cnn_recall': clf.get('recall'),
            'cnn_f1': clf.get('f1'),
            'cnn_balanced_accuracy': clf.get('balanced_accuracy'),
            'cnn_roc_auc': clf.get('roc_auc'),
            'cnn_pr_auc': clf.get('pr_auc'),
            'cnn_test_loss': keras_m.get('loss'),
            'cnn_test_accuracy_keras': keras_m.get('accuracy'),
            'generalization_gap': m.get('generalization_gap_train_minus_val_acc'),
            'epochs_ran': m.get('epochs_ran'),
            'train_time_s': m.get('train_time_s'),
        })
    return pd.DataFrame(rows)


## Sentinel-2 - Etapas 1 a 5

Este bloco roda pipeline completo apenas para Sentinel-2 e libera memoria ao final.


In [ ]:
sentinel2_result = None
if RUN_TRAINING:
    sentinel2_result = run_sensor_pipeline(
        sensor_key='sentinel2',
        sensor_cfg=SENSORS['sentinel2'],
        data_cfg=DATA_CFG,
        train_cfg=TRAIN_CFG,
        codes_path=CODES_PATH,
        output_root=OUTPUT_ROOT,
        seed=SEED,
        show_plots=True,
    )

sentinel2_result


## Landsat 8/9 - Etapas 1 a 5

Este bloco roda pipeline completo apenas para Landsat 8/9 e libera memoria ao final.


In [ ]:
landsat89_result = None
if RUN_TRAINING:
    landsat89_result = run_sensor_pipeline(
        sensor_key='landsat89',
        sensor_cfg=SENSORS['landsat89'],
        data_cfg=DATA_CFG,
        train_cfg=TRAIN_CFG,
        codes_path=CODES_PATH,
        output_root=OUTPUT_ROOT,
        seed=SEED,
        show_plots=True,
    )

landsat89_result


## MODIS - Etapas 1 a 5

Este bloco roda pipeline completo apenas para MODIS e libera memoria ao final.


In [ ]:
modis_result = None
if RUN_TRAINING:
    modis_result = run_sensor_pipeline(
        sensor_key='modis',
        sensor_cfg=SENSORS['modis'],
        data_cfg=DATA_CFG,
        train_cfg=TRAIN_CFG,
        codes_path=CODES_PATH,
        output_root=OUTPUT_ROOT,
        seed=SEED,
        show_plots=True,
    )

modis_result


## Comparacao quantitativa final (CNN vs MLP Sprint 2)

Aqui a comparacao e feita lendo os resultados ja salvos em disco por satelite,
sem recarregar os datasets de treino.


In [ ]:
comparison_df = collect_saved_metrics(OUTPUT_ROOT, SENSORS)
mlp_baseline = load_mlp_baseline_metrics(MLP_BASELINE_PATH)

if not comparison_df.empty:
    comparison_df['mlp_accuracy_sprint2'] = mlp_baseline.get('accuracy')
    comparison_df['delta_acc_cnn_minus_mlp'] = comparison_df['cnn_accuracy'] - comparison_df['mlp_accuracy_sprint2']

display(comparison_df)

if not comparison_df.empty:
    plot_df = comparison_df[['sensor', 'cnn_accuracy', 'mlp_accuracy_sprint2']].melt(
        id_vars='sensor',
        var_name='modelo',
        value_name='accuracy',
    )
    plt.figure(figsize=(8, 4))
    sns.barplot(data=plot_df, x='sensor', y='accuracy', hue='modelo')
    plt.ylim(0, 1)
    plt.title('Acuracia CNN por satelite vs baseline MLP (Sprint 2)')
    plt.tight_layout()
    plt.show()


## Analise critica consolidada


In [ ]:
analysis_lines = []

if comparison_df.empty:
    analysis_lines.append('Nenhum resultado de sensor encontrado para consolidacao.')
else:
    for _, row in comparison_df.iterrows():
        sensor = row['sensor']
        gap = row.get('generalization_gap')
        if pd.notna(gap):
            if gap > 0.08:
                analysis_lines.append(f'{sensor}: indicio de overfitting (gap={gap:.4f}).')
            else:
                analysis_lines.append(f'{sensor}: gap treino-validacao controlado (gap={gap:.4f}).')

        if pd.notna(row.get('delta_acc_cnn_minus_mlp')):
            delta = row['delta_acc_cnn_minus_mlp']
            trend = 'ganho' if delta >= 0 else 'queda'
            analysis_lines.append(f'{sensor}: {trend} de {delta:+.4f} em acuracia vs MLP Sprint 2.')

    if comparison_df['cnn_f1'].notna().any():
        best = comparison_df.sort_values('cnn_f1', ascending=False).iloc[0]
        analysis_lines.append(f"Melhor F1: {best['sensor']} ({best['cnn_f1']:.4f}).")

    analysis_lines.append(
        'Limitacao: o baseline MLP salvo no projeto possui poucas metricas detalhadas; '
        'a comparacao principal ficou concentrada em acuracia/loss.'
    )

for i, line in enumerate(analysis_lines, start=1):
    print(f'{i}. {line}')

comparison_path = OUTPUT_ROOT / 'cnn_multissatelite_comparison.csv'
analysis_path = OUTPUT_ROOT / 'cnn_multissatelite_analysis.txt'
summary_path = OUTPUT_ROOT / 'run_summary.json'

if not comparison_df.empty:
    comparison_df.to_csv(comparison_path, index=False)
analysis_path.write_text('\n'.join(analysis_lines), encoding='utf-8')

summary_payload = {
    'output_root': str(OUTPUT_ROOT),
    'sensors_with_results': comparison_df['sensor_key'].tolist() if not comparison_df.empty else [],
    'comparison_csv': str(comparison_path) if comparison_path.exists() else None,
    'analysis_txt': str(analysis_path),
}
save_json(summary_path, summary_payload)

print('\nArquivos finais:')
if comparison_path.exists():
    print('-', comparison_path)
print('-', analysis_path)
print('-', summary_path)
